<a href="https://colab.research.google.com/github/Lostvenom234/Experimental-Validation/blob/main/Experimental_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Role-, Weight-, and Information-Conditioned Generative State Transitions

**Framework:** CogMI  
**Implementation:** Python + FastAPI + Generative Model  
**Environment:** Google Colab  
**Status:** Research Prototype

---

### Overview

This notebook provides an experimental implementation of the CogMI
functional information framework.

The experiments investigate whether the generated state of a
generative system changes according to the interaction between:

- **Information content** \(I\)
- **Functional role** \(R\)
- **Influence weight** \(w\)

The conceptual transition is:

\[
G_{t+1}
=
\sigma
\left(
\sum_i w_{i,t}
\phi(I_{i,t},R_{i,t})
\right)
\]

The notebook further investigates recursive state transition:

\[
G_t + M_t + S_t
\rightarrow
G_{t+1}
\]

where:

- \(G_t\) = previous generative state
- \(M_t\) = memory information
- \(S_t\) = current sensory information

---

### Experimental Objective

The experiments are designed to isolate the behavioral effects of:

1. Information content
2. Functional role
3. Influence weight
4. Previous generative state

The objective is to examine whether changing these variables produces
corresponding changes in the generated state.

In [1]:
!pip -q install fastapi uvicorn pyngrok requests pydantic google-generativeai

In [2]:
import os
import requests

from dataclasses import dataclass
from typing import List

from pydantic import BaseModel, Field
from fastapi import FastAPI, HTTPException

from pyngrok import ngrok
import google.generativeai as genai

print("Imports successful.")

Imports successful.


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
!pip -q install -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 22.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.


In [4]:
# ============================================================
# CogMI Experimental Validation
# Imports
# ============================================================

import os
import requests

from dataclasses import dataclass
from typing import List

from pydantic import BaseModel, Field
from fastapi import FastAPI, HTTPException

from pyngrok import ngrok

from google import genai

print("Imports successful.")

Imports successful.


In [5]:
# ============================================================
# Secure API Configuration
# ============================================================

from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")

if not GEMINI_API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY is not configured in Colab Secrets."
    )

if not NGROK_AUTHTOKEN:
    raise RuntimeError(
        "NGROK_AUTHTOKEN is not configured in Colab Secrets."
    )

client = genai.Client(
    api_key=GEMINI_API_KEY
)

ngrok.set_auth_token(
    NGROK_AUTHTOKEN
)

print("Gemini client configured.")
print("Ngrok authentication configured.")

Gemini client configured.
Ngrok authentication configured.




Each incoming information element is represented by three independent
properties:

\[
E_i = (I_i, R_i, w_i)
\]

where:

- \(I_i\) = information content
- \(R_i\) = functional role
- \(w_i\) = influence weight

This separation allows the experiments to independently manipulate
information, role, and weight.

In [6]:
# ============================================================
# CogMI Information Element
# ============================================================

@dataclass
class InformationElement:
    """
    Represents one information element entering the CogMI system.

    I = information content
    R = functional role
    w = influence weight
    """

    content: str
    role: str
    weight: float


print("InformationElement defined successfully.")

InformationElement defined successfully.


In [7]:
# ============================================================
# Weight Validation
# ============================================================

def normalize_elements(
    elements: List[InformationElement]
) -> List[InformationElement]:

    if not elements:
        raise ValueError(
            "At least one information element is required."
        )

    if any(element.weight < 0 for element in elements):
        raise ValueError(
            "Weights cannot be negative."
        )

    total_weight = sum(
        element.weight
        for element in elements
    )

    if total_weight <= 0:
        raise ValueError(
            "Total weight must be greater than zero."
        )

    normalized = []

    for element in elements:

        normalized.append(
            InformationElement(
                content=element.content,
                role=element.role,
                weight=element.weight / total_weight
            )
        )

    return normalized

In [8]:
test_elements = [
    InformationElement(
        content="The object is a cake.",
        role="sensory",
        weight=0.8
    ),
    InformationElement(
        content="The object was previously identified as bread.",
        role="memory",
        weight=0.2
    )
]

normalized = normalize_elements(test_elements)

for element in normalized:
    print(
        element.role,
        "→",
        element.weight
    )

print(
    "Total:",
    sum(e.weight for e in normalized)
)

sensory → 0.8
memory → 0.2
Total: 1.0


## 3. Role-Conditioned Representation

The function

\[
\phi(I_i,R_i)
\]

represents information together with its functional role.

The same information can therefore enter the generative system through
different channels, such as:

- sensory
- memory
- previous generative state
- feedback

The role does not change the information content itself. It specifies
how the information should be treated during the state transition.

In [9]:
# ============================================================
# Role-Conditioned Representation
# ============================================================

def phi_role_conditioning(
    element: InformationElement
) -> str:

    return (
        f"[{element.role.upper()} CHANNEL | "
        f"WEIGHT={element.weight:.4f}]\n"
        f"{element.content}"
    )


print("Role-conditioned representation ready.")

Role-conditioned representation ready.


In [10]:
# Test role-conditioned representation

test_element = InformationElement(
    content="The object is a cake.",
    role="sensory",
    weight=0.8
)

print(
    phi_role_conditioning(test_element)
)

[SENSORY CHANNEL | WEIGHT=0.8000]
The object is a cake.


### Compiling the Information Stream

Multiple information elements are represented independently and then
compiled into a single structured context.

The compilation preserves:

\[
(I_i,R_i,w_i)
\]

for every incoming element.

In [11]:
# ============================================================
# Compile Information Stream
# ============================================================

def compile_information_stream(
    elements: List[InformationElement]
) -> str:

    normalized = normalize_elements(elements)

    blocks = [
        phi_role_conditioning(element)
        for element in normalized
    ]

    return "\n\n".join(blocks)

In [12]:
# Test multi-element compilation

test_elements = [
    InformationElement(
        content="The object is a cake.",
        role="sensory",
        weight=0.8
    ),
    InformationElement(
        content="The object was previously identified as bread.",
        role="memory",
        weight=0.2
    )
]

compiled = compile_information_stream(
    test_elements
)

print(compiled)

[SENSORY CHANNEL | WEIGHT=0.8000]
The object is a cake.

[MEMORY CHANNEL | WEIGHT=0.2000]
The object was previously identified as bread.


## 4. CogMI Generative State Transition

The generative transition receives a structured set of information
elements:

\[
E_i=(I_i,R_i,w_i)
\]

and constructs the next generative state:

\[
G_{t+1}
=
\sigma
\left(
\sum_i w_i\phi(I_i,R_i)
\right)
\]

The language model performs the semantic generation step, while the
Python implementation explicitly controls the information, functional
roles, and influence weights used in each experimental condition.

The model is instructed to treat the supplied weights as influence
parameters rather than modifying them.

In [13]:
# ============================================================
# CogMI Generative Transition
# ============================================================

MODEL_NAME = "gemini-2.5-flash"


def build_transition_prompt(
    elements: List[InformationElement]
) -> str:

    normalized = normalize_elements(elements)

    compiled_stream = compile_information_stream(
        normalized
    )

    weight_summary = "\n".join(
        [
            f"- {element.role}: "
            f"{element.weight:.4f}"
            for element in normalized
        ]
    )

    prompt = f"""
You are the generative transition component of an
experimental CogMI system.

The system represents each incoming information element as:

E_i = (I_i, R_i, w_i)

where:

I_i = information content
R_i = functional role
w_i = influence weight

The conceptual transition is:

G_(t+1) =
sigma(
    sum_i w_i * phi(I_i, R_i)
)

Your task is to generate the next state from the
provided information elements.

IMPORTANT:

1. Preserve the distinction between information content
   and functional role.

2. Treat the supplied weights as influence parameters.

3. Do not change the supplied weights.

4. Higher-weight information should have greater influence
   on the resulting state when information sources conflict.

5. Lower-weight information may still be acknowledged when
   relevant.

6. Do not invent information that is not present in the
   supplied elements.

CURRENT INFLUENCE WEIGHTS:

{weight_summary}

CURRENT INFORMATION STREAM:

{compiled_stream}

Generate the next generative state.

Return ONLY the resulting state text.
"""

    return prompt

In [14]:
# ============================================================
# Gemini State Generation
# ============================================================

def generate_next_state(
    elements: List[InformationElement]
) -> str:

    prompt = build_transition_prompt(
        elements
    )

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    if not response.text:
        raise RuntimeError(
            "Gemini returned an empty response."
        )

    return response.text.strip()

In [15]:
# ============================================================
# Transition Test
# ============================================================

test_elements = [

    InformationElement(
        content="The object is a cake.",
        role="sensory",
        weight=0.8
    ),

    InformationElement(
        content="The object was previously identified as bread.",
        role="memory",
        weight=0.2
    )
]

next_state = generate_next_state(
    test_elements
)

print("NEXT STATE:")
print(next_state)

NEXT STATE:
The object is a cake, though memory recalls a prior identification of bread.


## 5. Experiment 1 — Same Information, Different Roles

### Objective

This experiment examines whether functional role remains distinguishable
when the information content is identical.

Both information elements contain the same statement:

> "The object is a cake."

However, they enter the system through different functional roles:

- Sensory
- Memory

Both elements receive equal influence weight.

### Experimental condition

\[
I_S = I_M
\]

\[
R_S \neq R_M
\]

\[
w_S = w_M = 0.5
\]

The purpose is to isolate the effect of functional role while keeping
information content and influence weight constant.

In [18]:
# ============================================================
# Experiment Runner
# ============================================================

def run_experiment(
    name: str,
    elements: List[InformationElement]
):

    normalized = normalize_elements(elements)

    next_state = generate_next_state(
        normalized
    )

    print("=" * 70)
    print(name)
    print("=" * 70)

    print("\nINPUT ELEMENTS:")

    for element in normalized:
        print(f"\nRole: {element.role}")
        print(f"Weight: {element.weight:.4f}")
        print(f"Information: {element.content}")

    print("\nNEXT GENERATIVE STATE:")
    print(next_state)

    return {
        "experiment": name,
        "elements": [
            {
                "information": e.content,
                "role": e.role,
                "weight": e.weight
            }
            for e in normalized
        ],
        "next_state": next_state
    }

print("Experiment runner ready.")

Experiment runner ready.


In [ ]:
experiment_1 = run_experiment(
    "Experiment 1 — Same Information, Different Roles",
    [
        InformationElement(
            content="The object is a cake.",
            role="sensory",
            weight=0.5
        ),
        InformationElement(
            content="The object is a cake.",
            role="memory",
            weight=0.5
        )
    ]
)

Experiment 1 — Same Information, Different Roles

INPUT ELEMENTS:

Role: sensory
Weight: 0.5000
Information: The object is a cake.

Role: memory
Weight: 0.5000
Information: The object is a cake.

NEXT GENERATIVE STATE:
The object is confirmed to be a cake.


### Observation

Both information elements contain identical information:

\[
I_S = I_M
\]

The functional roles are different:

\[
R_S \neq R_M
\]

while the influence weights are equal:

\[
w_S = w_M = 0.5
\]

The generated state was:

> "The object is confirmed to be a cake."

The output remains consistent with the shared information content.

This experiment establishes the baseline condition in which role differs
but information content and influence weight are held constant.

## 6. Experiment 2 — Sensory Weight Dominance

### Objective

This experiment introduces conflicting information while assigning
greater influence to the sensory channel.

The sensory information indicates that the object is a cake, while
memory indicates that the object was previously identified as bread.

### Experimental condition

\[
I_S \neq I_M
\]

\[
R_S \neq R_M
\]

\[
w_S > w_M
\]

Specifically:

\[
w_S = 0.8,\qquad w_M = 0.2
\]

The experiment tests whether the higher-weight sensory information
dominates the resulting generative state while lower-weight memory
remains available to the transition.

In [19]:
experiment_2 = run_experiment(
    "Experiment 2 — Sensory Weight Dominance",
    [
        InformationElement(
            content="The object is a cake.",
            role="sensory",
            weight=0.8
        ),
        InformationElement(
            content="The object was previously identified as bread.",
            role="memory",
            weight=0.2
        )
    ]
)

Experiment 2 — Sensory Weight Dominance

INPUT ELEMENTS:

Role: sensory
Weight: 0.8000
Information: The object is a cake.

Role: memory
Weight: 0.2000
Information: The object was previously identified as bread.

NEXT GENERATIVE STATE:
The object is a cake, with a past memory suggesting it was bread.


### Experiment 2 — Observed Result

The previously executed experiment produced:

> "The object is a cake, despite having been previously identified as bread."

The experimental condition was:

\[
I_S \neq I_M
\]

\[
R_S \neq R_M
\]

\[
w_S = 0.8,\qquad w_M = 0.2
\]

The sensory information therefore had greater assigned influence than
the memory information.

### Observation

The generated state favored the sensory information while retaining
the conflicting memory information.

This result is consistent with the expected behavior of the
weight-controlled information integration condition.

## 7. Experiment 3 — Memory Weight Dominance

### Objective

This experiment reverses the influence relationship from Experiment 2.

The information remains conflicting:

- Sensory information: the object is a cake.
- Memory information: the object was previously identified as bread.

However, memory now receives the greater influence weight.

### Experimental condition

\[
I_S \neq I_M
\]

\[
R_S \neq R_M
\]

\[
w_M > w_S
\]

Specifically:

\[
w_S = 0.2857
\]

\[
w_M = 0.7143
\]

The purpose is to examine whether increasing the influence of memory
causes the resulting generative state to shift toward the historical
information.

In [20]:
# ============================================================
# Experiment 3 — Memory Weight Dominance
# ============================================================

experiment_3 = run_experiment(
    "Experiment 3 — Memory Weight Dominance",
    [
        InformationElement(
            content="The object is a cake.",
            role="sensory",
            weight=0.2857
        ),
        InformationElement(
            content="The object was previously identified as bread.",
            role="memory",
            weight=0.7143
        )
    ]
)

Experiment 3 — Memory Weight Dominance

INPUT ELEMENTS:

Role: sensory
Weight: 0.2857
Information: The object is a cake.

Role: memory
Weight: 0.7143
Information: The object was previously identified as bread.

NEXT GENERATIVE STATE:
The object is identified as bread, primarily informed by strong memory recall, though current sensory input suggests it is a cake.


### Observed Result

The previously executed experiment produced:

> "The object is bread, despite current sensory perception of it being a cake."

The experimental condition was:

\[
w_S = 0.2857
\]

\[
w_M = 0.7143
\]

Memory therefore had substantially greater influence than sensory
information.

### Observation

The resulting state shifted toward the memory information while still
acknowledging the conflicting sensory information.

This provides the complementary condition to Experiment 2.

## 8. Experiment 4 — Recursive Generative State Transition

### Objective

This experiment introduces the previous generative state together with
memory and current sensory information.

The transition is:

\[
G_t + M_t + S_t
\rightarrow
G_{t+1}
\]

The previous output is therefore no longer merely an observed result.
It becomes an information element entering the next recursive cycle.

### Experimental condition

Three information sources are provided:

- Previous generative state
- Memory
- Current sensory information

Each source has a distinct functional role and influence weight.

The experiment examines whether the resulting state reflects the
combined influence of the previous state, memory, and current sensory
information.

In [21]:
# ============================================================
# Experiment 4 — Recursive Generative State Transition
# ============================================================

def run_recursive_experiment(
    previous_state: str,
    memory: list[str],
    sensory: str,
    previous_weight: float,
    memory_weight: float,
    sensory_weight: float
):

    elements = [
        InformationElement(
            content=previous_state,
            role="previous",
            weight=previous_weight
        ),

        InformationElement(
            content=" | ".join(memory),
            role="memory",
            weight=memory_weight
        ),

        InformationElement(
            content=sensory,
            role="sensory",
            weight=sensory_weight
        )
    ]

    return run_experiment(
        "Experiment 4 — Recursive Generative State Transition",
        elements
    )

In [22]:
# ============================================================
# Recursive Cycle 1
# ============================================================

cycle_1 = run_recursive_experiment(
    previous_state="The object is bread.",

    memory=[
        "The object was previously identified as bread."
    ],

    sensory="The object is a cake.",

    previous_weight=0.3,
    memory_weight=0.2,
    sensory_weight=0.5
)

Experiment 4 — Recursive Generative State Transition

INPUT ELEMENTS:

Role: previous
Weight: 0.3000
Information: The object is bread.

Role: memory
Weight: 0.2000
Information: The object was previously identified as bread.

Role: sensory
Weight: 0.5000
Information: The object is a cake.

NEXT GENERATIVE STATE:
The object is a cake, despite previous information and memory suggesting it was bread.


In [23]:
# ============================================================
# Recursive Cycle 2
# ============================================================

cycle_2 = run_recursive_experiment(
    previous_state=cycle_1["next_state"],

    memory=[
        "The object was previously identified as bread."
    ],

    sensory="The object is confirmed to be a cake.",

    previous_weight=0.3,
    memory_weight=0.2,
    sensory_weight=0.5
)

Experiment 4 — Recursive Generative State Transition

INPUT ELEMENTS:

Role: previous
Weight: 0.3000
Information: The object is a cake, despite previous information and memory suggesting it was bread.

Role: memory
Weight: 0.2000
Information: The object was previously identified as bread.

Role: sensory
Weight: 0.5000
Information: The object is confirmed to be a cake.

NEXT GENERATIVE STATE:
The object is confirmed to be a cake, despite previous information and memory suggesting it was bread.


In [24]:
# ============================================================
# Recursive Cycle 2
# ============================================================

cycle_2 = run_recursive_experiment(
    previous_state=cycle_1["next_state"],

    memory=[
        "The object was previously identified as bread."
    ],

    sensory="The object is confirmed to be a cake.",

    previous_weight=0.3,
    memory_weight=0.2,
    sensory_weight=0.5
)

Experiment 4 — Recursive Generative State Transition

INPUT ELEMENTS:

Role: previous
Weight: 0.3000
Information: The object is a cake, despite previous information and memory suggesting it was bread.

Role: memory
Weight: 0.2000
Information: The object was previously identified as bread.

Role: sensory
Weight: 0.5000
Information: The object is confirmed to be a cake.

NEXT GENERATIVE STATE:
The object is confirmed to be a cake, despite previous information and memory suggesting it was bread.


# 9. Cross-Experiment Analysis

The four experiments progressively manipulate the three variables of
the CogMI information representation:

\[
E_i=(I_i,R_i,w_i)
\]

The experiments are designed to isolate the effects of information
content, functional role, influence weight, and recursive state.

---

## Experiment 1 — Role Baseline

Condition:

\[
I_S=I_M
\]

\[
R_S\neq R_M
\]

\[
w_S=w_M
\]

The information content was identical across sensory and memory
channels, while the functional roles differed and the weights were
equal.

The generated state remained consistent with the shared information.

---

## Experiment 2 — Sensory Weight Dominance

Condition:

\[
I_S\neq I_M
\]

\[
R_S\neq R_M
\]

\[
w_S>w_M
\]

with:

\[
w_S=0.8,\qquad w_M=0.2
\]

The generated state favored the sensory information while retaining
the conflicting memory information.

---

## Experiment 3 — Memory Weight Dominance

Condition:

\[
I_S\neq I_M
\]

\[
R_S\neq R_M
\]

\[
w_M>w_S
\]

with:

\[
w_S=0.2857,\qquad w_M=0.7143
\]

The generated state shifted toward the memory information while still
acknowledging the conflicting sensory information.

---

## Experiment 4 — Recursive State Transition

Condition:

\[
G_t+M_t+S_t
\rightarrow
G_{t+1}
\]

The previous generated state was introduced as an additional
information element in subsequent cycles.

This experiment therefore extends the static information integration
conditions into a recursive sequence of generative states.

In [ ]:
# ============================================================
# Cross-Experiment Summary
# ============================================================

experiment_summary = [
    {
        "Experiment": "1",
        "Information": "Same",
        "Role": "Different",
        "Weight": "Equal",
        "Purpose": "Role baseline"
    },
    {
        "Experiment": "2",
        "Information": "Different",
        "Role": "Different",
        "Weight": "Sensory > Memory",
        "Purpose": "Sensory dominance"
    },
    {
        "Experiment": "3",
        "Information": "Different",
        "Role": "Different",
        "Weight": "Memory > Sensory",
        "Purpose": "Memory dominance"
    },
    {
        "Experiment": "4",
        "Information": "Previous + Memory + Sensory",
        "Role": "Different",
        "Weight": "Controlled",
        "Purpose": "Recursive transition"
    }
]

for experiment in experiment_summary:
    print(
        f"Experiment {experiment['Experiment']}: "
        f"{experiment['Purpose']}"
    )

# 10. Experimental Findings

Across the controlled experiments, the generated state changed
according to the configuration of information, functional role, and
influence weight.

The complementary weight conditions provide an important comparison:

\[
w_S>w_M
\]

produced a sensory-dominant state, whereas:

\[
w_M>w_S
\]

produced a memory-dominant state.

The recursive experiment further introduces the previous generative
state as an information source for the subsequent transition.

These observations provide an implementation-level demonstration of the
CogMI functional representation:

\[
E_i=(I_i,R_i,w_i)
\]

and its recursive extension:

\[
G_{t+1}
=
\sigma
\left(
\sum_i w_i\phi(I_i,R_i)
\right).
\]

The experiments should be interpreted as controlled behavioral
demonstrations of the proposed framework rather than as proof that a
particular language model internally implements the mathematical
mechanism.

# 11. Reproducibility Notes

This notebook contains the implementation and experimental conditions
used to investigate the CogMI functional information framework.

The experiments use a generative language model as the semantic
generation component.

The following variables are explicitly controlled by the experimental
implementation:

- Information content
- Functional role
- Influence weight
- Previous generative state

API credentials are intentionally not included in this notebook.
They must be supplied through the execution environment.

Because generative model outputs may vary across executions, exact
wording of generated states may differ between runs. The experimental
analysis therefore focuses on the directional behavior of the generated
state under controlled information and weight conditions rather than
requiring identical textual output.